In [0]:
dbutils.widgets.text("catalog", "dbr_dev_ua5816bd")
dbutils.widgets.text("schema_landing", "roksolana_shendiu770")
dbutils.widgets.text("schema_bronze", "roksolana_shendiu770_bronze")

CATALOG = dbutils.widgets.get("catalog")
SCHEMA_LANDING = dbutils.widgets.get("schema_landing")
SCHEMA_BRONZE = dbutils.widgets.get("schema_bronze")

SOURCE_PATH = f"/Volumes/{CATALOG}/{SCHEMA_LANDING}/bronze_landing/petroleum_consumption"
SCHEMA_LOCATION = f"/Volumes/{CATALOG}/{SCHEMA_LANDING}/bronze_landing/_schemas/petroleum_consumption"
CHECKPOINT_LOCATION = f"/Volumes/{CATALOG}/{SCHEMA_LANDING}/bronze_landing/_checkpoints/petroleum_consumption"
TARGET_TABLE = f"{CATALOG}.{SCHEMA_BRONZE}.petroleum_consumption_bronze"

In [0]:
from pyspark.sql.functions import col

raw_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", SCHEMA_LOCATION)
    .option("cloudFiles.inferColumnTypes", "true")
    .option("cloudFiles.schemaEvolutionMode", "addNewColumnsWithTypeWidening")
    .option("cloudFiles.maxFilesPerTrigger", 100)
    .load(SOURCE_PATH)
    .select("*", col("_metadata.file_path").alias("source_file_path"))
)

query = (
    raw_stream.writeStream
    .option("checkpointLocation", CHECKPOINT_LOCATION)
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .toTable(TARGET_TABLE)
)

query.awaitTermination()